# Stage 2.5 queue ownership, batching, underfoot insertion, and hiring economics

**Execution status: NOT RUN.** This is a pinned-source Kaggle handoff only; no numbers in this checked-in notebook are executed results.

The evaluator compares three candidate-only arms against the same stochastic P-final policy and deterministic frozen BC-E opponent. BC-E is frozen on `E_LEGACY` with the committed `standard_mixed` opening. The full panel is the established ordered 32-seed panel, both seat orientations. The four named cases are a supplemental, explicitly identity-preserving capture slice: contrast seeds `1392524882`, `814255690`; regression-watch seeds `229020805`, `1962565411`; both orientations. They are reported per seed and are not blanket failures.

Only A vs B and B vs C are produced. There is no A-vs-C comparison and no combined-leader selection. Direct A confirmation is required before selecting any combined leader. The existing queue+scheduling fallback remains in the pinned source; this artifact does not edit executor code, tests, or sharder implementation.

In [ ]:
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
import gzip, hashlib, json, math, os, random, re, subprocess, sys, time, zipfile

REPO_URL = 'https://github.com/BillXu21/Kaggriculture.git'
EXPERIMENT_REF = os.environ.get('KAGGRICULTURE_EXPERIMENT_REF', 'main')
SOURCE_SHA = os.environ.get('KAGGRICULTURE_SOURCE_SHA', 'REPLACE_WITH_40_HEX_SOURCE_SHA')
PPO_CHECKPOINT = Path(os.environ.get('PPO_CHECKPOINT', 'REPLACE_WITH_PPO_CHECKPOINT_PATH'))
BC_E_CHECKPOINT = Path(os.environ.get('BC_E_CHECKPOINT', 'REPLACE_WITH_BC_E_CHECKPOINT_PATH'))
MASTER_SEED, BACKEND, E_HISTORY_VERSION, OPENING = 25, 'fast', 'E_LEGACY', 'standard_mixed'
PROCESSES, THREADS_PER_PROCESS = 4, 1
VARIANTS = ['combined', 'combined_wheat3']
STARVATION_WORKLOAD_VISIBILITY_REPAIR = False
RESUME_RUNS = False
PANEL_SEEDS = [144368101, 309507, 615013, 918079, 1221109, 1524137, 1827169, 2130193, 2433221, 2736251, 3039283, 3342311, 3645341, 3948373, 4251401, 2112243121, 1470672056, 995106988, 1303793286, 521973470, 107449192, 768565387, 1370134739, 2090797777, 425789796, 1027359148, 1688475343, 261654734, 863224086, 1524340281, 97519672, 699089024]
CONTRAST_SEEDS = [1392524882, 814255690]
REGRESSION_SEEDS = [229020805, 1962565411]
TARGET_SEEDS = CONTRAST_SEEDS + REGRESSION_SEEDS
CAPTURE_SEEDS = PANEL_SEEDS + TARGET_SEEDS
if len(PANEL_SEEDS) != 32 or len(set(PANEL_SEEDS)) != 32: raise ValueError('PANEL_SEEDS must remain the established 32-seed order')
if len(set(CAPTURE_SEEDS)) != 36: raise ValueError('capture seed list must be 32 panel plus four supplemental cases')
COMMON_FLAGS = ['--backend', BACKEND, '--e-history-version', E_HISTORY_VERSION, '--master-seed', str(MASTER_SEED), '--processes', str(PROCESSES), '--variants', *VARIANTS, '--underfoot-first', '--deadline-safe-planting', '--deadline-safe-hiring']
ARM_FLAGS = {
    'A': ['--persistent-worker-queues', '--schedule-informed-hiring', '--schedule-hiring-economic-repair'],
    'B': ['--persistent-worker-queues', '--queue-ownership-repair', '--batch-reserved-supplies', '--underfoot-queue-insertion', '--schedule-informed-hiring', '--schedule-hiring-economic-repair'],
    'C': ['--persistent-worker-queues', '--queue-ownership-repair', '--batch-reserved-supplies', '--underfoot-queue-insertion', '--schedule-informed-hiring', '--schedule-hiring-economic-repair', '--suppress-expansion-from-prior-debt', 'off'],
}
if STARVATION_WORKLOAD_VISIBILITY_REPAIR: raise AssertionError('starvation visibility must remain OFF')
RUN_ROOT = Path('/kaggle/working') / ('stage25_queue_batch_hiring_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
RUN_ROOT.mkdir(parents=True, exist_ok=False); REPO = RUN_ROOT / 'repo'
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''): digest.update(chunk)
    return digest.hexdigest()
def episode_id_for(seeds, seed, seat): return MASTER_SEED * (2 * len(seeds)) + 2 * list(seeds).index(int(seed)) + int(seat)
if not re.fullmatch(r'[0-9a-fA-F]{40}', SOURCE_SHA): raise RuntimeError('Set KAGGRICULTURE_SOURCE_SHA to the exact pushed 40-hex source SHA')
for path, label in ((PPO_CHECKPOINT, 'PPO_CHECKPOINT'), (BC_E_CHECKPOINT, 'BC_E_CHECKPOINT')):
    if not path.is_file(): raise FileNotFoundError(f'{label} is not mounted: {path}')


In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not token: raise RuntimeError('Kaggle Secret GITHUB_TOKEN is empty')
try:
    with __import__('tempfile').TemporaryDirectory(prefix='stage25_askpass_') as tmp:
        askpass = Path(tmp) / 'askpass.py'
        askpass.write_text("import os, sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ['STAGE25_TOKEN'])\n")
        env = {**os.environ, 'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'STAGE25_TOKEN': token}
        subprocess.run(['git', 'clone', '--depth', '1', '--no-checkout', REPO_URL, str(REPO)], env=env, check=True, timeout=180)
        subprocess.run(['git', 'fetch', '--depth', '1', 'origin', SOURCE_SHA], cwd=REPO, env=env, check=True, timeout=180)
        subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO, env=env, check=True, timeout=30)
finally:
    token = None
    if 'env' in globals(): env.pop('STAGE25_TOKEN', None)
actual_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if actual_sha.lower() != SOURCE_SHA.lower(): raise RuntimeError(f'checkout mismatch: {actual_sha} != {SOURCE_SHA}')
if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip(): raise RuntimeError('pinned source checkout is dirty')
tracked = [REPO / 'executor_v0/agent.py', REPO / 'executor_v0/foreman.py', REPO / 'executor_v0/scheduler.py', REPO / 'tools/evaluate_stage25_upkeep.py', REPO / 'tools/run_stage25_upkeep_sharded.py']
provenance = {'source_sha_requested': SOURCE_SHA, 'source_sha_checked_out': actual_sha, 'source_hashes': {str(p.relative_to(REPO)): sha256(p) for p in tracked}, 'ppo_sha256': sha256(PPO_CHECKPOINT), 'bc_e_sha256': sha256(BC_E_CHECKPOINT), 'backend': BACKEND, 'e_history_version': E_HISTORY_VERSION, 'opening': OPENING, 'processes': PROCESSES, 'threads_per_process': THREADS_PER_PROCESS, 'panel_seeds': PANEL_SEEDS, 'capture_seeds': CAPTURE_SEEDS, 'starvation_workload_visibility_repair': False}
(RUN_ROOT / 'provenance.json').write_text(json.dumps(provenance, indent=2) + '\n')
def preflight(name, seeds, flags, filters=None):
    output = RUN_ROOT / (name + '_preflight')
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--preflight-only', '--output-dir', str(output), '--seeds', *map(str, seeds), *COMMON_FLAGS, *flags]
    if filters: command += ['--game-filters', *filters]
    subprocess.run(command, cwd=REPO, env={**os.environ, 'PYTHONPATH': str(REPO)}, check=True)
    manifest = json.loads((output / 'manifest.json').read_text())
    if manifest['ordered_seeds'] != list(seeds) or manifest['config'].get('processes') != PROCESSES or manifest['config'].get('starvation_workload_visibility_repair', False): raise RuntimeError(f'preflight mismatch: {name}')
    return manifest
for arm, flags in ARM_FLAGS.items():
    preflight(arm + '_panel', PANEL_SEEDS, flags)
    preflight(arm + '_targeted', CAPTURE_SEEDS, flags, [f'{seed}:{seat}' for seed in TARGET_SEEDS for seat in (0, 1)])
subprocess.run([sys.executable, '-c', 'import bc_manager_jax, rl_manager, fast_env'], cwd=REPO, env={**os.environ, 'PYTHONPATH': str(REPO)}, check=True)
print('Pinned-source preflight passed; no evaluation result is claimed by this artifact.')


In [ ]:
EVAL_ENV = os.environ.copy(); EVAL_ENV.update({'PYTHONUNBUFFERED': '1', 'OMP_NUM_THREADS': '1', 'MKL_NUM_THREADS': '1', 'OPENBLAS_NUM_THREADS': '1', 'NUMEXPR_NUM_THREADS': '1', 'XLA_PYTHON_CLIENT_PREALLOCATE': 'false', 'XLA_FLAGS': '--xla_cpu_multi_thread_eigen=false intra_op_parallelism_threads=1', 'JAX_PLATFORMS': 'cpu', 'PYTHONPATH': str(REPO)})
def run_arm(arm, seeds, capture=False, filters=None, resume=RESUME_RUNS):
    kind = 'targeted' if capture else 'panel'; name = f'{arm}_{kind}'; output = RUN_ROOT / name
    capture_dir = RUN_ROOT / (name + '_captures') if capture else None
    command = [sys.executable, '-m', 'tools.run_stage25_upkeep_sharded', '--checkpoint', str(PPO_CHECKPOINT), '--e-checkpoint', str(BC_E_CHECKPOINT), '--output-dir', str(output), '--seeds', *map(str, seeds), *COMMON_FLAGS, *ARM_FLAGS[arm]]
    if capture_dir is not None: command += ['--capture-dir', str(capture_dir)]
    if filters: command += ['--game-filters', *filters]
    if resume: command += ['--resume']
    (RUN_ROOT / (name + '.command.json')).write_text(json.dumps({'arm': arm, 'kind': kind, 'command': command, 'source_sha': actual_sha, 'ordered_seeds': list(seeds), 'filters': filters, 'capture_passive': True}, indent=2) + '\n')
    started = time.monotonic(); log_path = RUN_ROOT / (name + '.log')
    with log_path.open('w') as log:
        process = subprocess.Popen(command, cwd=REPO, env=EVAL_ENV, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
        for line in process.stdout: log.write(line); log.flush(); print(line, end='', flush=True)
        code = process.wait()
    (RUN_ROOT / (name + '.timing.json')).write_text(json.dumps({'arm': arm, 'kind': kind, 'elapsed_seconds': time.monotonic() - started, 'resume_requested': bool(resume), 'completed': code == 0}, indent=2) + '\n')
    if code: raise RuntimeError(f'{name} failed with {code}; partial outputs remain at {output}')
    return output, capture_dir
TARGET_FILTERS = [f'{seed}:{seat}' for seed in TARGET_SEEDS for seat in (0, 1)]
target_outputs, panel_outputs = {}, {}
for arm in ('A', 'B', 'C'): target_outputs[arm] = run_arm(arm, CAPTURE_SEEDS, capture=True, filters=TARGET_FILTERS)
for arm in ('A', 'B', 'C'): panel_outputs[arm] = run_arm(arm, PANEL_SEEDS)
print('Requested commands completed only if this cell was run; the checked-in handoff remains a no-run artifact.')


In [ ]:
def load_rows(output):
    manifest = json.loads((output / 'manifest.json').read_text()); rows = [json.loads(x) for x in (output / 'games.jsonl').read_text().splitlines() if x.strip()]; return manifest, rows
def validate_output(output, seeds, expected_pairs):
    manifest, rows = load_rows(output); config = manifest.get('config', {})
    if manifest.get('status') != 'complete' or manifest.get('ordered_seeds') != list(seeds) or len(rows) != expected_pairs * len(VARIANTS): raise RuntimeError(f'incomplete output: {output}')
    for key, expected in {'backend': BACKEND, 'e_history_version': E_HISTORY_VERSION, 'processes': PROCESSES, 'variants': VARIANTS, 'starvation_workload_visibility_repair': False}.items():
        if config.get(key) != expected: raise RuntimeError(f'{output}: {key} mismatch')
    seen = set()
    for row in rows:
        identity = (row['variant'], int(row['seed']), int(row['seat']), int(row['episode_id']))
        if int(row['episode_id']) != episode_id_for(seeds, row['seed'], row['seat']) or identity in seen: raise RuntimeError(f'identity mismatch: {identity}')
        seen.add(identity)
    return manifest, rows
panel_data = {arm: validate_output(panel_outputs[arm][0], PANEL_SEEDS, len(PANEL_SEEDS) * 2) for arm in ('A', 'B', 'C')}
target_data = {arm: validate_output(target_outputs[arm][0], CAPTURE_SEEDS, len(TARGET_SEEDS) * 2) for arm in ('A', 'B', 'C')}
def pct(values, p):
    if not values: return None
    values = sorted(values); pos = (len(values) - 1) * p; lo, hi = math.floor(pos), math.ceil(pos)
    return values[lo] if lo == hi else values[lo] + (values[hi] - values[lo]) * (pos - lo)
def cluster_ci(rows, field):
    grouped = defaultdict(list)
    for row in rows: grouped[int(row['seed'])].append(float(row[field]))
    seeds = sorted(grouped); observed = [v for seed in seeds for v in grouped[seed]]; rng = random.Random(2509); reps = []
    for _ in range(4000):
        sample = [rng.choice(seeds) for _ in seeds]; values = [v for seed in sample for v in grouped[seed]]; reps.append(sum(values) / len(values))
    return {'mean': sum(observed) / len(observed), 'ci95': [pct(reps, .025), pct(reps, .975)], 'clusters': len(seeds), 'cluster_unit': 'seed; both seats and variants retained'}
def arm_summary(rows):
    wlt = {'W': 0, 'L': 0, 'T': 0}
    for row in rows: wlt['W' if row['margin'] > 0 else 'L' if row['margin'] < 0 else 'T'] += 1
    out = {'games': len(rows), 'wlt': wlt}
    for field in ('bank', 'opponent_bank', 'margin'): out[field] = cluster_ci(rows, field)
    out['win_rate'] = cluster_ci([{**row, 'win': 1 if row['margin'] > 0 else 0} for row in rows], 'win'); return out
def keyed(rows): return {(str(x['variant']), int(x['seed']), int(x['seat'])): x for x in rows}
def pair_summary(left_rows, right_rows, left_name, right_name):
    left, right = keyed(left_rows), keyed(right_rows); paired = []
    for key in sorted(set(left) & set(right)):
        a, b = left[key], right[key]; paired.append({'variant': key[0], 'seed': key[1], 'seat': key[2], 'episode_id': int(a['episode_id']), 'delta_bank': float(b['bank']) - float(a['bank']), 'delta_opponent_bank': float(b['opponent_bank']) - float(a['opponent_bank']), 'delta_margin': float(b['margin']) - float(a['margin'])})
    out = {'left': left_name, 'right': right_name, 'paired_games': len(paired), 'right_over_left_wlt': {'W': sum(x['delta_margin'] > 0 for x in paired), 'L': sum(x['delta_margin'] < 0 for x in paired), 'T': sum(x['delta_margin'] == 0 for x in paired)}, 'paired_rows': paired}
    for field in ('delta_bank', 'delta_opponent_bank', 'delta_margin'): out[field] = cluster_ci(paired, field)
    return out
report = {'panel': {arm: arm_summary(panel_data[arm][1]) for arm in ('A', 'B', 'C')}, 'comparisons': {'A_vs_B': pair_summary(panel_data['A'][1], panel_data['B'][1], 'A', 'B'), 'B_vs_C': pair_summary(panel_data['B'][1], panel_data['C'][1], 'B', 'C')}, 'pairwise_comparisons_only': ['A_vs_B', 'B_vs_C']}
report['targeted_cases'] = {str(seed): {arm: arm_summary([x for x in target_data[arm][1] if int(x['seed']) == seed]) for arm in ('A', 'B', 'C')} for seed in TARGET_SEEDS}
(RUN_ROOT / 'competitive_report.json').write_text(json.dumps(report, indent=2, allow_nan=False) + '\n'); print(json.dumps({k: v['right_over_left_wlt'] for k, v in report['comparisons'].items()}, indent=2))


In [ ]:
def read_gz(path):
    with gzip.open(path, 'rb') as stream: return json.loads(stream.read().decode('utf-8'))
def counts(state, seat):
    crops, animals = Counter(), Counter(); farms = state.get('farms') or []; tiles = farms[seat].get('tiles', []) if seat < len(farms) else []
    for row in tiles:
        for tile in row:
            if isinstance(tile, dict) and isinstance(tile.get('animal'), str): animals[tile['animal']] += 1
            elif isinstance(tile, dict) and tile.get('kind') == 'PLANT' and isinstance(tile.get('crop'), str): crops[tile['crop']] += 1
    return dict(crops), dict(animals)
def detailed_metrics(capture_root, arm):
    metrics, profile_inputs = [], []
    for meta_path in sorted(Path(capture_root).glob('*/*/meta.json')):
        directory = meta_path.parent; meta = json.loads(meta_path.read_text()); trace = read_gz(directory / 'debug_trace.json.gz'); candidate = int(meta['candidate_seat'])
        movement = productive = submitted_hires = 0; missed = 0; suppressed = 0; prior_suppressed = 0; current_suppressed = 0; pickups = []; final_crops = {}; final_animals = {}; missing_hire_cost = True; missing_missed = True; missing_suppression = True;
        for turn in trace.get('turns', []):
            state = turn.get('canonical_state') or {}; final_crops, final_animals = counts(state, candidate); joint = turn.get('joint_actions') or {}; actions = joint.get(str(candidate)) or {}; workers = [actions.get('farmer') or ['PASS']] + list(actions.get('hands') or [])
            for action in workers:
                op = str(action[0]) if action else 'PASS'
                if op in {'NORTH', 'SOUTH', 'EAST', 'WEST'}: movement += 1
                elif op == 'PICKUP' and len(action) >= 3 and isinstance(action[2], (int, float)): pickups.append(int(action[2]))
                elif op not in {'PASS', 'DROP'}: productive += 1
            submitted_hires += sum(1 for order in actions.get('market') or [] if order and order[0] == 'HIRE')
            debug = (turn.get('executor_debug') or {}).get(str(candidate)) or {}; survival = debug.get('survival') or {};
            if 'expansion_suppressed' in survival: suppressed += int(bool(survival['expansion_suppressed'])); missing_suppression = False
            for prof_seat in (0, 1):
                prof_debug = (turn.get('executor_debug') or {}).get(str(prof_seat))
                if prof_debug is not None: profile_inputs.append({'schema_version': 1, 'input_kind': 'executor_call_replay_input', 'arm': arm, 'variant': meta['variant'], 'seed': int(meta['seed']), 'seat': int(meta['seat']), 'episode_id': int(meta['episode_id']), 'executor_seat': prof_seat, 'step': int(turn['step']), 'day': int(turn['day']), 'hour': int(turn['hour']), 'canonical_state': state, 'executor_debug': prof_debug, 'joint_actions': turn.get('joint_actions')})
        diagnostics = read_gz(directory / f'executor_seat{candidate}.json.gz')
        for record in (diagnostics.get('days') or {}).values():
            labor = record.get('previous_labor') or {}; missed_value = record.get('missed_maintenance')
            if isinstance(labor.get('hire_cost'), (int, float)): missing_hire_cost = False
            if isinstance(missed_value, (list, tuple, set, dict)): missed += len(missed_value); missing_missed = False
            for item in (record.get('turn_trace') or []):
                expansion = item.get('expansion') or {}; prior_suppressed += int(bool(expansion.get('suppressed_from_prior'))); current_suppressed += int(bool(expansion.get('suppressed_current'))); missing_suppression = False
            submitted_hires += 0;
        hire_spending = sum(float((record.get('previous_labor') or {}).get('hire_cost', 0.0)) for record in (diagnostics.get('days') or {}).values() if isinstance((record.get('previous_labor') or {}).get('hire_cost', 0.0), (int, float)))
        metrics.append({'arm': arm, 'variant': meta['variant'], 'seed': int(meta['seed']), 'seat': int(meta['seat']), 'episode_id': int(meta['episode_id']), 'bank': float(meta['final_banks'][candidate]), 'opponent_bank': float(meta['final_banks'][1 - candidate]), 'hiring_expense_actual': hire_spending if not missing_hire_cost else None, 'submitted_hires': submitted_hires, 'pickup_batch_sizes': pickups, 'pickup_batch_count': len(pickups), 'pickup_batch_mean': sum(pickups) / len(pickups) if pickups else None, 'pickup_batch_max': max(pickups) if pickups else None, 'movement': movement, 'productive_work_actions': productive, 'missed_maintenance': missed if not missing_missed else None, 'final_crops': final_crops, 'final_animals': final_animals, 'expansion_suppressed_turns': suppressed if not missing_suppression else None, 'expansion_suppressed_from_prior_debt': prior_suppressed if not missing_suppression else None, 'expansion_suppressed_current': current_suppressed if not missing_suppression else None})
    return metrics, profile_inputs
detail_rows, profile_rows = [], []
for arm in ('A', 'B', 'C'):
    rows, inputs = detailed_metrics(target_outputs[arm][1], arm); detail_rows.extend(rows); profile_rows.extend(inputs)
if len(detail_rows) != 48 or not profile_rows: raise RuntimeError('targeted capture/profile-input coverage is incomplete')
with (RUN_ROOT / 'per_call_profile_inputs.jsonl').open('w', encoding='utf-8') as stream:
    for row in profile_rows: stream.write(json.dumps(row, sort_keys=True, allow_nan=False) + '\n')
(RUN_ROOT / 'per_call_profile_inputs.manifest.json').write_text(json.dumps({'schema_version': 1, 'records': len(profile_rows), 'source': 'targeted debug_trace.json.gz executor_debug', 'coverage': 'A/B/C x both variants x four target seeds x both orientations x both executor seats', 'note': 'inputs only; no cProfile run or performance claim'}, indent=2) + '\n')
runtime = {name: json.loads((RUN_ROOT / (name + '.timing.json')).read_text()) for name in ['A_targeted', 'B_targeted', 'C_targeted', 'A_panel', 'B_panel', 'C_panel']}
report_path = RUN_ROOT / 'competitive_report.json'; report = json.loads(report_path.read_text()); report['evaluator_runtime'] = runtime; report_path.write_text(json.dumps(report, indent=2, allow_nan=False) + '\n')
(RUN_ROOT / 'target_metrics.json').write_text(json.dumps({'status': 'generated only after execution', 'rows': detail_rows, 'evaluator_runtime': runtime, 'metrics': ['hiring_expense_actual', 'pickup_batch_sizes', 'movement', 'productive_work_actions', 'missed_maintenance', 'final_crops', 'final_animals', 'expansion_suppressed_turns', 'expansion_suppressed_from_prior_debt', 'evaluator_runtime']}, indent=2, allow_nan=False) + '\n')
archive = RUN_ROOT / 'stage25_queue_batch_hiring_handoff.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in sorted(RUN_ROOT.rglob('*')):
        if path.is_file() and path != archive and REPO not in path.parents: bundle.write(path, path.relative_to(RUN_ROOT).as_posix())
print(f'Wrote ZIP with traces and per-call profiling inputs: {archive}')


## Operator gate

Verify every manifest has the same source/checkpoint hashes, fast-engine provenance, `E_LEGACY`, `standard_mixed`, four processes, bounded one-thread child configuration, ordered seeds, both orientations, and successful resume validation. Read the four supplemental seeds individually: contrasts are diagnostic contrasts, and regression-watch cases are not automatic promotion vetoes.

The report must retain both banks, margin/W-L-T, seed-clustered uncertainty, actual hire spending, observed pickup quantities/batch sizes, movement, productive work, missed maintenance, crop/animal counts, expansion-suppression totals and causes, evaluator wall runtime, and fallback errors. Missing telemetry stays unavailable rather than becoming zero. The ZIP must contain complete debug traces and `per_call_profile_inputs.jsonl`; those are profiling inputs only.

Do not select a combined leader until A is directly confirmed by the operator after reviewing A’s full panel and targeted captures.